# ⚡ Entraînement — Classifieur Pokémon Gen 1

**Partie de Marwan** — architecture du modèle et pipeline d'entraînement.

Ce notebook déroule les deux checkpoints de `docs/PARTIE_MARWAN.md` :

| Checkpoint | Objectif | Commit attendu |
|---|---|---|
| 1 | Le pipeline tourne, 1 epoch complet, loss affichée | `feat: pipeline de base, premier epoch` |
| 2 | Modèle convergé, **val accuracy > 70%** | `feat: modèle convergé, accuracy X%` |

**Avant de lancer** : `Exécution → Modifier le type d'exécution → GPU T4`.
Sans GPU, comptez plus d'une heure au lieu de ~25 minutes.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi -L || echo "AUCUN GPU — passez en T4 : Execution > Modifier le type d execution > GPU"

## 2. Cloner le repo sur la branche `marwan`

In [ ]:
import os

REPO = "https://github.com/Hicham77500/Marwan-Khalil-Hicham-projet-dl.git"
BRANCH = "marwan"

if not os.path.exists("/content/projet-dl"):
    !git clone --branch {BRANCH} {REPO} /content/projet-dl
else:
    !cd /content/projet-dl && git fetch origin && git checkout {BRANCH} && git pull

%cd /content/projet-dl
!git log --oneline -5

## 3. Installer les dépendances

Colab embarque déjà TensorFlow. On installe le reste (`tf2onnx`, `kaggle`…).

In [ ]:
!pip install -q -r requirements-train.txt

import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU détecté :", tf.config.list_physical_devices("GPU") or "AUCUN")

## 4. Credentials Kaggle

Sur [kaggle.com](https://www.kaggle.com) → photo de profil → **Settings** → section **API**.
Kaggle propose deux formats, la cellule gère les deux :

| Format | Ce que vous obtenez | Ce que vous faites |
|---|---|---|
| **Actuel** (*Create New Token*) | une chaîne `KGAT_...` | collez-la quand la cellule la demande |
| **Legacy** (*Legacy API Credentials*) | un fichier `kaggle.json` | uploadez-le quand la cellule le propose |

> Le token est saisi en **entrée masquée** : il n'apparaît pas dans le notebook et
> ne partira donc pas sur GitHub si vous sauvegardez ce fichier.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
token_file = kaggle_dir / "access_token"
json_file = kaggle_dir / "kaggle.json"

if not token_file.exists() and not json_file.exists():
    print("Collez votre token KGAT_... (rien ne s'affichera a la saisie).")
    print("Laissez VIDE et validez pour uploader un kaggle.json a la place.")
    token = getpass("Token Kaggle : ").strip()

    if token:
        token_file.write_text(token)
        token_file.chmod(0o600)
    else:
        from google.colab import files
        import shutil
        uploaded = files.upload()
        shutil.move("kaggle.json", json_file)
        json_file.chmod(0o600)

# Le client lit ~/.kaggle/access_token tout seul, mais l'env var evite
# toute ambiguite si les deux fichiers coexistent.
if token_file.exists():
    os.environ["KAGGLE_API_TOKEN"] = token_file.read_text().strip()

print("access_token :", token_file.exists(), "| kaggle.json :", json_file.exists())

### Vérifier l'authentification

Mieux vaut échouer ici, en 2 secondes, qu'au milieu du téléchargement.
Un token `KGAT_` exige le client **kaggle >= 2.0** : les versions 1.x, que Colab
préinstalle, l'ignorent silencieusement et renvoient une erreur d'authentification.

In [ ]:
!kaggle --version
!kaggle datasets files lantian773030/pokemonclassification | head -5

## 5. Télécharger et normaliser le dataset

Le dataset Kaggle se décompresse dans un sous-dossier (`PokemonData/`), et
certains miroirs séparent les images en `train/`, `valid/`, `test/`. Or
`src/data_loader.py` attend **`data/raw/<Classe>/*.jpg`**.

La cellule ci-dessous aplatit n'importe laquelle de ces structures vers le
format attendu : elle regroupe les images par **nom du dossier parent**, ce qui
fusionne au passage les splits (le `data_loader` refait son propre découpage
train/val/test de façon stratifiée).

In [ ]:
!kaggle datasets download -d lantian773030/pokemonclassification -p data/raw --unzip --force

In [ ]:
from collections import defaultdict
from pathlib import Path
import shutil

RAW = Path("data/raw")
EXT = {".jpg", ".jpeg", ".png", ".gif", ".webp"}
# Dossiers de split : ce ne sont pas des noms de classes.
SPLITS = {"train", "training", "valid", "validation", "val", "test", "images", "data"}


def normalize(raw: Path) -> dict:
    """Aplatit l'arborescence vers raw/<classe>/*.jpg et retourne la distribution."""
    groups = defaultdict(list)
    for path in raw.rglob("*"):
        if path.is_file() and path.suffix.lower() in EXT:
            cls = path.parent.name
            if cls.lower() in SPLITS:
                continue  # image posée directement dans un split : pas de classe
            groups[cls].append(path)

    moved = 0
    for cls, paths in groups.items():
        target_dir = raw / cls
        target_dir.mkdir(parents=True, exist_ok=True)
        for src in paths:
            if src.parent == target_dir:
                continue
            dest = target_dir / src.name
            i = 1
            while dest.exists():  # collision entre deux splits
                dest = target_dir / f"{src.stem}_{i}{src.suffix}"
                i += 1
            shutil.move(str(src), str(dest))
            moved += 1

    # Nettoyer les dossiers vides laissés derrière
    for path in sorted(raw.rglob("*"), key=lambda p: -len(p.parts)):
        if path.is_dir() and not any(path.iterdir()):
            path.rmdir()

    return {cls: len(p) for cls, p in groups.items()}, moved


distribution, moved = normalize(RAW)
print(f"{moved} fichiers deplaces")
print(f"{len(distribution)} classes, {sum(distribution.values())} images")
print("Exemples :", sorted(distribution)[:5])
print("Min/classe :", min(distribution.values()), "| Max/classe :", max(distribution.values()))

## 6. Inspecter le dataset (checkpoint technique du PDF)

« Dataset chargé et inspecté : shapes, exemples, distribution des classes ».

In [ ]:
from src.data_loader import inspect_dataset

info = inspect_dataset()

In [ ]:
# Aperçu visuel : une image par classe pour les 10 premières classes
import matplotlib.pyplot as plt
from PIL import Image
from src.data_loader import get_data_paths

paths, labels = get_data_paths()
seen, samples = set(), []
for p, l in zip(paths, labels):
    if l not in seen:
        seen.add(l)
        samples.append((p, l))
    if len(samples) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (p, l) in zip(axes.ravel(), samples):
    ax.imshow(Image.open(p).convert("RGB"))
    ax.set_title(l, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Checkpoint 1 — le pipeline tourne

2 epochs, sans fine-tuning. L'accuracy sera basse : **ce n'est pas le sujet**.
On vérifie que les données chargent, que le modèle se compile, et que la loss descend.

In [ ]:
!python -m src.train --epochs 2 --no-finetune

### Commit du checkpoint 1

À lancer depuis **votre machine** (le repo local), pas depuis Colab :

```bash
git add models/training_results.json
git commit -m "feat: pipeline de base, premier epoch"
```

## 8. Checkpoint 2 — entraînement complet

Deux phases : 10 epochs base gelée, puis 5 epochs de fine-tuning à LR/10.
Comptez ~25 minutes sur T4.

Objectif : **val accuracy > 70%**, contre un baseline aléatoire de 0.66%.

In [ ]:
import time

start = time.time()
!python -m src.train
print(f"\nDuree totale : {(time.time() - start) / 60:.1f} min")

In [ ]:
import json

results = json.load(open("models/training_results.json"))
for k, v in results.items():
    print(f"{k:24} {v}")

acc = results["test_accuracy"]
baseline = results["baseline"]
print(f"\nTest accuracy : {acc:.2%}")
print(f"Baseline      : {baseline:.2%}  ->  {acc / baseline:.0f}x mieux")
print("Objectif 70%  :", "ATTEINT" if acc > 0.70 else "NON ATTEINT")

## 9. Courbes d'entraînement (TensorBoard)

Pour la soutenance : « courbes loss/val ». Comparez `phase1` et `phase2`.
Si `val_loss` remonte pendant que `loss` descend → overfitting.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 10. Export ONNX pour la WebApp

`app.py` charge un `.onnx`, pas le `.keras`. **Sans cette étape, la démo continue
de servir l'ancien modèle** même après un réentraînement.

Le script vérifie que le nombre de sorties correspond à `class_names.txt`.

In [ ]:
!python -m src.export_onnx

## 11. Récupérer les artefacts

Téléchargez les fichiers, placez-les dans `models/` du repo local, puis commitez.

In [ ]:
from google.colab import files

for f in [
    "models/pokemon_classifier.keras",
    "models/pokemon_classifier.onnx",
    "models/class_names.txt",
    "models/training_results.json",
]:
    files.download(f)

### Commit du checkpoint 2

Depuis votre machine, une fois les fichiers copiés dans `models/` :

```bash
git add models/pokemon_classifier.keras models/pokemon_classifier.onnx
git add models/class_names.txt models/training_results.json
git commit -m "feat: modele converge, accuracy X%"
```

Puis mettre à jour le tableau **Résultats** du README avec les vrais chiffres.